# GNT end-to-end example notebook

This notebook demonstrates an end-to-end GNT workflow for TF-target gene interaction prediction, including data loading, representation learning, edge embedding construction, SVM-based classification, and evaluation.

## 1. Imports

The following cell imports the required Python packages and project modules used for loading data, training the GNT model, constructing edge embeddings, and evaluating classifier performance.

In [ ]:
# Standard library imports
import os

# Third-party scientific computing libraries
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score, average_precision_score
from tqdm import tqdm

# Project-specific modules
import LoadData as data
from convertdata import *
from GNT import GNT
from evaluation import *
from utils import *


## 2. Model hyperparameters

These parameters define the embedding dimensions, the relative contribution of attribute embeddings, the number of negative samples, and the optimization settings used during GNT training.

In [ ]:
# Hyperparameter configuration for GNT
parameters = {
    'id_embedding_size': 128,
    'attr_embedding_size': 128,
    'representation_size': 128,
    'alpha': 1,
    'n_neg_samples': 10,
    'epoch': 30,
    'batch_size': 256,
    'learning_rate': 0.002
}

# Display the parameter dictionary for verification
parameters


## 3. Classifier training function

After learning node embeddings with GNT, edge embeddings are passed to an SVM classifier. The following function performs hyperparameter selection using GridSearchCV with ROC-AUC as the optimization criterion.

In [ ]:
# Define the classifier and hyperparameter grid for edge classification
def train_classifier(train_data, train_labels):
    # Use a balanced SVM so that class imbalance is handled during optimization
    clf = SVC(class_weight='balanced', probability=True)

    # Search over the regularization parameter C
    clf_parameters = {
        'clf__C': [1, 90]
    }

    # Wrap the classifier in a pipeline for compatibility with GridSearchCV
    pipeline = Pipeline([('clf', clf)])

    # Perform cross-validation and select the model with the best ROC-AUC
    grid = GridSearchCV(pipeline, clf_parameters, scoring='roc_auc', cv=10)
    grid.fit(train_data, train_labels)

    return grid


## 4. Data loading, GNT training, and evaluation

The next function loads one dataset, trains the GNT model on positive and negative training edges, constructs edge embeddings, trains the downstream SVM classifier, and evaluates performance on the test set using AUC-ROC and AUPR.

In [ ]:
# Load one dataset, train GNT, and evaluate the downstream classifier
def load_and_train_model(base_dir, dataset_type, sub_folder, main_folder):
    # Build the dataset-specific directory path
    directory_path = os.path.join(base_dir, dataset_type, sub_folder, main_folder)

    # Read the gene identifier file used to map learned embeddings back to genes
    gene_id_file = pd.read_csv(os.path.join(directory_path, 'gene_ID.tsv'), sep='\t')

    # Define input feature and edge-list files
    feature_file = os.path.join(directory_path, 'expression_values.csv')
    link_file = os.path.join(directory_path, 'edgelist.csv')

    # Load positive and negative train/test edge splits
    train_edges = np.load(os.path.join(directory_path, 'train_edges.npy'))
    train_edges_false = np.load(os.path.join(directory_path, 'train_edges_false.npy'))
    test_edges = np.load(os.path.join(directory_path, 'test_edges.npy'))
    test_edges_false = np.load(os.path.join(directory_path, 'test_edges_false.npy'))

    print('#####')

    # Initialize the dataset object required by GNT
    Data = data.LoadData(directory_path + '/', train_links=train_edges, features_file=feature_file)

    # Initialize the GNT model with a fixed random seed
    model = GNT('', Data, 2018, parameters)

    # Combine positive and negative edges for supervised representation learning
    training_edges = np.concatenate([train_edges, train_edges_false])
    train_edge_labels = np.concatenate([np.ones(len(train_edges)), np.zeros(len(train_edges_false))])

    # Train the GNT model and obtain node embeddings
    embeddings, attr_embeddings = model.train(training_edges, train_edge_labels)

    # Create a mapping from gene identifiers to learned embeddings
    gene_ids = gene_id_file['GeneName']
    gene_embeddings_dict = {gene_ids[i]: embeddings[i] for i in range(len(gene_ids))}

    # Construct edge embeddings for positive and negative training samples
    pos_train_edge_embs = get_edge_embeddings(embeddings, train_edges)
    neg_train_edge_embs = get_edge_embeddings(embeddings, train_edges_false)
    train_edge_embs = np.concatenate([pos_train_edge_embs, neg_train_edge_embs])
    train_edge_labels = np.concatenate([np.ones(len(train_edges)), np.zeros(len(train_edges_false))])

    # Shuffle the edge embeddings before classifier training
    index = np.random.permutation(len(train_edge_labels))
    train_data = train_edge_embs[index, :]
    train_labels = train_edge_labels[index]

    # Train the SVM classifier with cross-validated hyperparameter selection
    grid = train_classifier(train_data, train_labels)

    # Construct test edge embeddings for evaluation
    pos_test_edge_embs = get_edge_embeddings(embeddings, test_edges)
    neg_test_edge_embs = get_edge_embeddings(embeddings, test_edges_false)
    test_edge_embs = np.concatenate([pos_test_edge_embs, neg_test_edge_embs])

    # Create ground-truth test labels and generate classifier scores
    test_edge_labels = np.concatenate([np.ones(len(test_edges)), np.zeros(len(test_edges_false))])
    test_preds = grid.predict_proba(test_edge_embs)[:, 1]

    # Compute evaluation metrics
    test_roc = roc_auc_score(test_edge_labels, test_preds)
    test_ap = average_precision_score(test_edge_labels, test_preds)

    return test_roc, test_ap


## 5. Run experiments across selected datasets

This function iterates over the selected benchmark configuration, trains the model, evaluates it, and stores the resulting AUC-ROC and AUPR values in a results list.

In [ ]:
# Loop through the selected datasets and collect evaluation metrics
def run_experiments(base_dir):
    # Define the benchmark subsets to be evaluated
    sub_folders = ['hESC']
    dataset_types = ['Specific Dataset']
    main_folders = ['TFs+1000']

    results = []

    for dataset_type in dataset_types:
        for sub_folder in sub_folders:
            for main_folder in main_folders:
                print(f'Processing: {dataset_type} - {sub_folder} - {main_folder}')
                test_roc, test_ap = load_and_train_model(base_dir, dataset_type, sub_folder, main_folder)
                results.append({
                    'Dataset Type': dataset_type,
                    'Sub Folder': sub_folder,
                    'Main Folder': main_folder,
                    'AUC-ROC': test_roc,
                    'AUPR': test_ap
                })

    # Convert results to a DataFrame for easier inspection if needed
    # results_df = pd.DataFrame(results)
    # results_df.to_csv('GNT_specific_results.csv', index=False)

    print(results)
    return results


## 6. Execute the pipeline

The final cell defines the benchmark dataset directory and launches the experiment for the selected configuration.

In [ ]:
# Set the root directory containing the BEELINE benchmark datasets
base_dir = 'beeline_dataset/Benchmark Dataset'

# Run the experiment
results = run_experiments(base_dir)
